In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.sparse import issparse
from scipy.special import rel_entr

from pyscf import gto, scf, fci
from qiskit.quantum_info import SparsePauliOp, Statevector, state_fidelity
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import ADAM, COBYLA, SPSA
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.transformers import FreezeCoreTransformer
from qiskit_nature.second_q.circuit.library import HartreeFock

In [2]:
MOLECULE = "H2"
BASIS = "sto-3g"
R = 0.96  

GEOM = f"H 0 0 0; H 0 0 {R}"

mol = gto.M(atom=GEOM, basis=BASIS, unit="Angstrom")
mf = scf.RHF(mol).run(verbose=0)
E_HF, E_FCI = mf.e_tot, fci.FCI(mf).kernel()[0]

print(f"{MOLECULE}/{BASIS} R={R}")
print(f"E_HF  = {E_HF:.6f} Ha")
print(f"E_FCI = {E_FCI:.6f} Ha")
print(f"Corr  = {(E_FCI-E_HF)*1000:.2f} mHa")

H2/sto-3g R=0.96
E_HF  = -1.077020 Ha
E_FCI = -1.109367 Ha
Corr  = -32.35 mHa


In [3]:
problem = PySCFDriver(atom=GEOM, basis=BASIS, unit=DistanceUnit.ANGSTROM).run()
prob_fc = FreezeCoreTransformer().transform(problem)
mapper = prob_fc.get_tapered_mapper(mapper=JordanWignerMapper())

qop = mapper.map(prob_fc.hamiltonian.second_q_op())
enuc = prob_fc.hamiltonian.nuclear_repulsion_energy
hf_circuit = HartreeFock(prob_fc.num_spatial_orbitals, prob_fc.num_particles, mapper)
e_off = problem.reference_energy - float(np.real(Statevector(hf_circuit).expectation_value(qop))) - enuc

n_sys = qop.num_qubits
print(f"Original Qubbits: {problem.num_spin_orbitals}, Terms: {len(prob_fc.hamiltonian.second_q_op())}")
print(f"Qubits: {n_sys}, Terms: {len(qop)}")

Original Qubbits: 4, Terms: 36
Qubits: 1, Terms: 3


In [5]:
problem = PySCFDriver(atom=GEOM, basis=BASIS, unit=DistanceUnit.ANGSTROM).run()
prob_fc = FreezeCoreTransformer().transform(problem)

print("FULL")
print("  num_spin_orbitals :", problem.num_spin_orbitals)
print("  num_particles     :", problem.num_particles)

print("FREEZE-CORE")
print("  num_spin_orbitals :", prob_fc.num_spin_orbitals)
print("  num_particles     :", prob_fc.num_particles)

# no tapering
mapper_jw = JordanWignerMapper()
qop_jw = mapper_jw.map(prob_fc.hamiltonian.second_q_op())

print("JW NO TAPER")
print("  qop.num_qubits    :", qop_jw.num_qubits)
print("  len(qop_jw)       :", len(qop_jw))

# tapered
mapper_taper = prob_fc.get_tapered_mapper(mapper=JordanWignerMapper())
qop_taper = mapper_taper.map(prob_fc.hamiltonian.second_q_op())

print("JW TAPERED")
print("  qop.num_qubits    :", qop_taper.num_qubits)
print("  len(qop_taper)    :", len(qop_taper))

FULL
  num_spin_orbitals : 4
  num_particles     : (1, 1)
FREEZE-CORE
  num_spin_orbitals : 4
  num_particles     : (1, 1)
JW NO TAPER
  qop.num_qubits    : 4
  len(qop_jw)       : 15
JW TAPERED
  qop.num_qubits    : 1
  len(qop_taper)    : 3


In [4]:
def build_haa(n_sys, n_anc, n_layers, hf_circuit=None, internal="can"):
    if internal == "can":
        n_params = 3 * n_sys * (n_layers + 1) + 3 * n_sys * n_anc * n_layers
    else:
        n_params = 3 * n_sys * (n_layers + 1) + 6 * n_sys * n_anc * n_layers

    p = ParameterVector("θ", n_params)
    qc = QuantumCircuit(n_sys + n_anc)

    if hf_circuit:
        qc.compose(hf_circuit, qubits=range(n_sys), inplace=True)

    i = 0

    for q in range(n_sys):
        qc.u(p[i], p[i+1], p[i+2], q)
        i += 3

    for _ in range(n_layers):
        for a in range(n_anc):
            for s in range(n_sys):
                anc = n_sys + a

                if internal == "can":
                    qc.rxx(p[i], s, anc)
                    qc.ryy(p[i+1], s, anc)
                    qc.rzz(p[i+2], s, anc)
                    i += 3

                elif internal == "u3cx":
                    qc.u(p[i], p[i+1], p[i+2], s)
                    qc.u(p[i+3], p[i+4], p[i+5], anc)
                    qc.cx(s, anc)
                    i += 6

        for q in range(n_sys):
            qc.u(p[i], p[i+1], p[i+2], q)
            i += 3

    return qc

In [5]:
def build_qrqnn(
    n_sys: int,
    n_anc: int,
    n_layers: int,
    hf_circuit: QuantumCircuit | None = None
) -> QuantumCircuit:
    total_qubits = n_sys + n_anc

    n_params = 3 * total_qubits * (n_layers + 1) + 3 * n_sys * n_anc * n_layers
    params = ParameterVector("t", n_params)

    qc = QuantumCircuit(total_qubits, name="QRQNN")

    if hf_circuit is not None:
        qc.compose(hf_circuit, qubits=range(n_sys), inplace=True)

    idx = 0

    for q in range(total_qubits):
        qc.u(params[idx], params[idx + 1], params[idx + 2], q)
        idx += 3

    for _ in range(n_layers):
        if n_sys > 1:
            for q in range(n_sys - 1):
                qc.cx(q, q + 1)
            qc.cx(n_sys - 1, 0)

        for anc in range(n_anc):
            anc_q = n_sys + anc
            for sys_q in range(n_sys):
                qc.rxx(params[idx], sys_q, anc_q)
                qc.ryy(params[idx + 1], sys_q, anc_q)
                qc.rzz(params[idx + 2], sys_q, anc_q)
                idx += 3

        for q in range(total_qubits):
            qc.u(params[idx], params[idx + 1], params[idx + 2], q)
            idx += 3

    return qc

In [6]:
from scipy.stats import entropy


def expressibility(circuit, n_samples=500, n_bins=50, eps=1e-12, seed=None):
    n_params = circuit.num_parameters
    n_qubits = circuit.num_qubits
    N = 2 ** n_qubits
    rng = np.random.default_rng(seed)

    fidelities = []

    for _ in range(n_samples):
        theta1 = rng.uniform(0, 2 * np.pi, n_params)
        theta2 = rng.uniform(0, 2 * np.pi, n_params)

        psi1 = Statevector(circuit.assign_parameters(theta1))
        psi2 = Statevector(circuit.assign_parameters(theta2))

        fidelities.append(state_fidelity(psi1, psi2))

    fidelities = np.array(fidelities)

    hist, edges = np.histogram(fidelities, bins=n_bins, range=(0, 1), density=True)
    width = edges[1] - edges[0]

    p_ansatz = hist * width

    p_haar = (1 - edges[:-1]) ** (N - 1) - (1 - edges[1:]) ** (N - 1)

    p_ansatz = np.clip(p_ansatz, eps, None)
    p_haar = np.clip(p_haar, eps, None)

    p_ansatz /= np.sum(p_ansatz)
    p_haar /= np.sum(p_haar)

    return entropy(p_ansatz, p_haar)

In [7]:
def bp_analysis(circuit, qop, n_anc, n_samples=20):
    qop_ext = SparsePauliOp.from_list([("I"*n_anc,1.0)]).tensor(qop) if n_anc else qop
    H = qop_ext.to_matrix(); H = H.toarray() if issparse(H) else H
    grads = []
    for _ in range(n_samples):
        params = np.random.uniform(0, 2*np.pi, circuit.num_parameters)
        g = []
        for i in range(min(10, circuit.num_parameters)):
            pp, pm = params.copy(), params.copy(); pp[i] += np.pi/2; pm[i] -= np.pi/2
            psip, psim = Statevector(circuit.assign_parameters(pp)).data, Statevector(circuit.assign_parameters(pm)).data
            g.append((np.real(np.vdot(psip, H@psip)) - np.real(np.vdot(psim, H@psim))) / 2)
        grads.append(g)
    return np.mean(np.abs(grads))

In [ ]:
def run_vqe(circuit, qop, optimizer, n_anc=0, n_restart=5, enuc=0.0, e_off=0.0):
    qop_ext = SparsePauliOp.from_list([("I"*n_anc,1.0)]).tensor(qop) if n_anc else qop
    qop_shift = qop_ext + SparsePauliOp.from_list([("I"*qop_ext.num_qubits, enuc + e_off)])
    best_energy = np.inf
    for _ in range(n_restart): 
        vqe = VQE(StatevectorEstimator(), circuit, optimizer, initial_point=np.random.uniform(0, 2*np.pi, circuit.num_parameters))
        energy = vqe.compute_minimum_eigenvalue(qop_shift).eigenvalue.real
        best_energy = min(best_energy, energy)
    return best_energy

In [ ]:
mapper_taper = prob_fc.get_tapered_mapper(mapper=JordanWignerMapper())
max_iter = 900
qop_taper = mapper_taper.map(prob_fc.hamiltonian.second_q_op())

haa_circuit = build_haa(n_sys, n_anc=1, n_layers=1, hf_circuit=hf_circuit)
hea_circuit = build_haa(n_sys, n_anc=0, n_layers=1, hf_circuit=hf_circuit, internal="u3cx")
qrqnn_circuit = build_qrqnn(n_sys, n_anc=1, n_layers=1, hf_circuit=hf_circuit)

print("Expressibility:")
print("  HAA   :", expressibility(haa_circuit, n_samples=1000))
print("  HEA   :", expressibility(hea_circuit, n_samples=1000))
print("  QRQNN :", expressibility(qrqnn_circuit, n_samples=1000))

print("\nBP Analysis:")
print("  HAA   :", bp_analysis(haa_circuit, qop_taper, n_anc=1))
print("  HEA   :", bp_analysis(hea_circuit, qop_taper, n_anc=0))
print("  QRQNN :", bp_analysis(qrqnn_circuit, qop_taper, n_anc=1))

haa_energy = run_vqe(haa_circuit, qop_taper, ADAM(maxiter=max_iter), n_anc=1, enuc=enuc, e_off=e_off)
hea_energy = run_vqe(hea_circuit, qop_taper, ADAM(maxiter=max_iter), n_anc=0, enuc=enuc, e_off=e_off)
qrqnn_energy = run_vqe(qrqnn_circuit, qop_taper, ADAM(maxiter=max_iter), n_anc=1, enuc=enuc, e_off=e_off)

print("\nVQE Energies:")
print(f"  HAA   : {haa_energy:.6f} Ha")
print(f"  HEA   : {hea_energy:.6f} Ha")
print(f"  QRQNN : {qrqnn_energy:.6f} Ha")

print("\nEnergy Errors (mHa):")
print(f"  HAA   : {(haa_energy - E_FCI)*1000:.2f} mHa")
print(f"  HEA   : {(hea_energy - E_FCI)*1000:.2f} mHa")
print(f"  QRQNN : {(qrqnn_energy - E_FCI)*1000:.2f} mHa")


Expressibility:
  HAA   : 0.032165889200887636
  HEA   : 0.030016532684918943
  QRQNN : 0.015506634083632932

BP Analysis:
  HAA   : 0.12488528411747964
  HEA   : 0.15409381854856702
  QRQNN : 0.13448402156952124

VQE Energies:
  HAA   : -1.109367 Ha
  HEA   : -1.109367 Ha
  QRQNN : -1.109367 Ha

Energy Errors (mHa):
  HAA   : 0.00 mHa
  HEA   : 0.00 mHa
  QRQNN : 0.00 mHa


In [ ]:
def get_molecule(R):
    return f"H 0 0 0; H 0 0 {R}"

def generate_pec(minimum, maximum, step):
    distances = np.arange(minimum, maximum + step, step)
    energies = []
    for R in distances:
        geom = get_molecule(R)
        mol = gto.M(atom=geom, basis=BASIS, unit="Angstrom")
        mf = scf.RHF(mol).run(verbose=0)
        E_HF, E_FCI = mf.e_tot, fci.FCI(mf).kernel()[0]

        
        problem = PySCFDriver(atom=geom, basis=BASIS, unit=DistanceUnit.ANGSTROM).run()
        prob_fc = FreezeCoreTransformer().transform(problem)
        mapper = prob_fc.get_tapered_mapper(mapper=JordanWignerMapper())

        qop = mapper.map(prob_fc.hamiltonian.second_q_op())
        enuc = prob_fc.hamiltonian.nuclear_repulsion_energy
        hf_circuit = HartreeFock(prob_fc.num_spatial_orbitals, prob_fc.num_particles, mapper)
        e_off = problem.reference_energy - float(np.real(Statevector(hf_circuit).expectation_value(qop))) - enuc

        n_sys = qop.num_qubits

        e_haa = run_vqe(build_haa(n_sys, n_anc=1, n_layers=1, hf_circuit=hf_circuit), qop_taper, ADAM(maxiter=max_iter), n_anc=1, enuc=enuc, e_off=e_off)
        e_hea = run_vqe(build_haa(n_sys, n_anc=0, n_layers=1, hf_circuit=hf_circuit,internal="u3cx"), qop_taper, ADAM(maxiter=max_iter), n_anc=0, enuc=enuc, e_off=e_off)
        e_qrqnn = run_vqe(build_qrqnn(n_sys, n_anc=1, n_layers=1, hf_circuit=hf_circuit), qop_taper, ADAM(maxiter=max_iter), n_anc=1, enuc=enuc, e_off=e_off)
        energies.append((R, E_HF, E_FCI, e_haa, e_hea, e_qrqnn))
    return energies

In [17]:
import pandas as pd

df = pd.DataFrame(generate_pec(0.5, 3.0, 0.1), columns=["R", "E_HF", "E_FCI", "E_HAA", "E_HEA", "E_QRQNN"])
df

KeyboardInterrupt: 